In [ ]:
import pandas as pd
import geopandas as gpd
from os import path, makedirs

# Carregando os dados extraídos no notebook anterior

Neste notebook, vamos utilizar os dados extraídos e salvos pelo notebook `13 area de parques - extração.ipynb`.

In [ ]:
cache_dir = path.join('data', 'cache', 'urbanismo')

In [ ]:
filename = path.join(cache_dir, 'subprefeituras.gpkg')
subprefeituras = gpd.read_file(filename, layer='subprefeituras')
subprefeituras

In [ ]:
filename = path.join(cache_dir, 'parques.gpkg')
parques = gpd.read_file(filename, layer='parques')
parques

# Calculando as áreas de parque por subprefeitura

## Conferindo os CRS

In [ ]:
subprefeituras.crs

In [ ]:
parques.crs

## Calculando as intereseções de parques e subprefeituras

In [ ]:
intersect = subprefeituras.overlay(parques[~parques['tx_data_inauguracao'].str.contains('Proposto')], how='intersection')
intersect

In [ ]:
intersect['area_m2'] = intersect['geometry'].area
intersect

In [ ]:
area_parques_por_subprefeitura = (
    intersect
    .groupby('nm_subprefeitura')
    .agg({'area_m2': 'sum'})
    .rename(columns={'area_m2': 'qt_area_parques_metro'})
    .reset_index()
)
area_parques_por_subprefeitura

In [ ]:
subprefeituras_parques = subprefeituras.merge(
    area_parques_por_subprefeitura,
    how='left',
    left_on='nm_subprefeitura',
    right_on='nm_subprefeitura'
)
subprefeituras_parques

In [ ]:
subprefeituras_parques['pct_area_parques'] = (
    subprefeituras_parques['qt_area_parques_metro'] /
    subprefeituras_parques['qt_area_metro']
    * 100
)
subprefeituras_parques

In [ ]:
100*subprefeituras_parques['qt_area_parques_metro'].sum()/subprefeituras_parques['qt_area_metro'].sum()

In [ ]:
subprefeituras_parques.plot(column='pct_area_parques',
                            legend=True,
                            cmap='Greens')